In [1]:
# Fetch the SRGAN code
!git clone https://github.com/Lornatang/SRGAN-PyTorch.git

# Move into the folder
%cd SRGAN-PyTorch

# Note: We are skipping the requirements.txt because Kaggle 
# already has a perfectly optimized PyTorch environment built-in!

Cloning into 'SRGAN-PyTorch'...
remote: Enumerating objects: 3631, done.
remote: Counting objects: 100% (918/918), done.
remote: Compressing objects: 100% (277/277), done.
remote: Total 3631 (delta 688), reused 814 (delta 637), pack-reused 2713 (from 1)
Receiving objects: 100% (3631/3631), 2.98 MiB | 15.55 MiB/s, done.
Resolving deltas: 100% (2385/2385), done.
/kaggle/working/SRGAN-PyTorch


In [2]:
import os
import yaml

config_path = './configs/train/SRGAN_x4-ImageNet.yaml'

with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

# 1. 90-Epoch Lock
cfg['TRAIN']['HYP']['EPOCHS'] = 90
cfg['TRAIN']['LR_SCHEDULER']['MILESTONES'] = [45, 68]

# 2. Map SAR Datasets
base_path = "/kaggle/input/datasets/maryclauds/sar-image-patches/SAR_Dataset/SAR_Dataset"
cfg['TRAIN']['DATASET']['TRAIN_GT_IMAGES_DIR'] = f"{base_path}/train_HR"
cfg['TRAIN']['DATASET']['TRAIN_LR_IMAGES_DIR'] = f"{base_path}/train_LR"
cfg['TEST']['DATASET']['PAIRED_TEST_GT_IMAGES_DIR'] = f"{base_path}/train_HR"
cfg['TEST']['DATASET']['PAIRED_TEST_LR_IMAGES_DIR'] = f"{base_path}/train_LR"

# 3. Start from Phase 1 Weights
cfg['TRAIN']['CHECKPOINT']['PRETRAINED_G_MODEL'] = "/kaggle/input/datasets/maryclauds/sar-image-patches/epoch_90.pth.tar"
cfg['TRAIN']['CHECKPOINT']['PRETRAINED_D_MODEL'] = ""
cfg['TRAIN']['CHECKPOINT']['RESUMED_G_MODEL'] = ""
cfg['TRAIN']['CHECKPOINT']['RESUMED_D_MODEL'] = ""

# 4. Fix Kaggle GPU compiled flags
cfg['MODEL']['G']['COMPILED'] = False
cfg['MODEL']['D']['COMPILED'] = False
cfg['MODEL']['EMA']['COMPILED'] = False

with open(config_path, 'w') as f:
    yaml.dump(cfg, f)

# --- THE HOT-PATCHES ---
utils_path = 'utils.py'
with open(utils_path, 'r') as f:
    utils_code = f.read()

# Fix the Scheduler Resume Bug
utils_code = utils_code.replace(
    'scheduler.load_state_dict(checkpoint["scheduler"])',
    'pass # [HOTFIX] Bypassed missing scheduler!'
)

# Fix the 20GB Disk Crash Bug (Prevent it from saving intermediate epoch trash)
utils_code = utils_code.replace(
    'torch.save(state_dict, checkpoint_path)',
    'if "epoch_" not in checkpoint_path: torch.save(state_dict, checkpoint_path)\n    else: pass # [HOTFIX] Saved Kaggle disk space!'
)

with open(utils_path, 'w') as f:
    f.write(utils_code)

print("✅ Phase 1 weights mapped. 90 Epochs locked.")
print("✅ Source code patched: Disk Space crash neutralized!")

✅ Phase 1 weights mapped. 90 Epochs locked.
✅ Source code patched: Disk Space crash neutralized!


In [3]:
!python train_gan.py --config_path ./configs/train/SRGAN_x4-ImageNet.yaml

2026-03-17 07:51:37.012555: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773733897.444096      54 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773733897.564573      54 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773733898.803914      54 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773733898.803966      54 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773733898.803973      54 computation_placer.cc:177] computation placer alr